In [1]:
# ==========================================
# سلول 1: Import و Setup
# ==========================================
import os
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
import cohere
import re
from collections import Counter

# ===========================
# تنظیمات
# ===========================
QDRANT_URL = "http://localhost:6333"

COLLECTION_LAWS = "legal_laws"
COLLECTION_UNITY = "legal_votes_unity"
COLLECTION_DADNAMEH = "legal_votes_dadnameh"

MODEL_EMBED = "baai/bge-m3"

# ===========================
# API Keys
# ===========================
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_EMBEDDINGS_API_KEY")
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_EMBEDDINGS_API_KEY تنظیم نشده است")
if not COHERE_API_KEY:
    raise ValueError("COHERE_API_KEY تنظیم نشده است")

# ===========================
# اتصال به سرویس‌ها
# ===========================
qdrant = QdrantClient(url=QDRANT_URL)

client_embed = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

co = cohere.Client(api_key=COHERE_API_KEY)

print("✓ Qdrant متصل شد")
print("✓ Cohere آماده است")
print("✓ Embedding client آماده است")

# ===========================
# بارگذاری فایل‌های متادیتا
# ===========================
KEYWORDS_PATH = r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\keywords.txt"
QAVANIN_PATH = r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\qavanin_karbordi.txt"

# خواندن keywords
with open(KEYWORDS_PATH, "r", encoding="utf-8") as f:
    KEYWORDS = set(line.strip() for line in f if line.strip())

# خواندن قوانین کاربردی
with open(QAVANIN_PATH, "r", encoding="utf-8") as f:
    QAVANIN_TEXT = f.read()
    # استخراج نام قوانین (خطوطی که با "قانون" شروع می‌شوند)
    QAVANIN_LIST = [
        line.strip() 
        for line in QAVANIN_TEXT.split('\n') 
        if line.strip() and (line.strip().startswith('قانون') or line.strip().startswith('لایحه'))
    ]

print(f"✓ بارگذاری شد: {len(KEYWORDS)} keyword")
print(f"✓ بارگذاری شد: {len(QAVANIN_LIST)} قانون کاربردی")
print("\n" + "="*80)

✓ Qdrant متصل شد
✓ Cohere آماده است
✓ Embedding client آماده است
✓ بارگذاری شد: 76 keyword
✓ بارگذاری شد: 108 قانون کاربردی



In [2]:
import json
from collections import defaultdict

def extract_all_metadata_titles():
    """
    استخراج تمام تیترهای متادیتا از هر 3 کالکشن
    و ساخت یک reverse index برای جستجوی سریع
    """
    
    # ساختار داده:
    # keyword → [{law_name, metadata_type, metadata_value, full_text}, ...]
    reverse_index = defaultdict(list)
    
    # آمار
    stats = {
        "total_points": 0,
        # برای قوانین
        "volume_titles": 0,
        "book_titles": 0,
        "bab_titles": 0,
        "chapter_titles": 0,
        "section_titles": 0,
        "topic_titles": 0,
        "paragraph_titles": 0,
        # برای آراء
        "issuer_count": 0,
        "title_count": 0,
        "category_count": 0,
        "law_name_count": 0,
        "ruling_topic_count": 0
    }
    
    collections = [COLLECTION_LAWS, COLLECTION_UNITY, COLLECTION_DADNAMEH]
    
    for collection_name in collections:
        print(f"\n🔍 در حال پردازش {collection_name}...")
        
        offset = None
        batch_count = 0
        
        while True:
            result = qdrant.scroll(
                collection_name=collection_name,
                limit=100,
                offset=offset,
                with_payload=True,
                with_vectors=False
            )
            
            points, offset = result
            
            if not points:
                break
            
            for point in points:
                payload = point.payload
                law_name = payload.get("law_name", "نامشخص")
                
                # ===========================
                # بخش 1: پردازش قوانین (COLLECTION_LAWS)
                # ===========================
                if collection_name == COLLECTION_LAWS:
                    metadata_fields = [
                        ("volume", "جلد"),
                        ("book", "کتاب"),
                        ("bab", "باب"),
                        ("chapter", "فصل"),
                        ("section", "بخش"),
                        ("topic", "مبحث"),
                        ("paragraph", "فقره")
                    ]
                    
                    for field_name, field_type in metadata_fields:
                        if field_name in payload and payload[field_name]:
                            full_text = str(payload[field_name]).strip()
                            
                            if not full_text or len(full_text) < 3:
                                continue
                            
                            # استخراج تیتر (بعد از "-" یا ":")
                            title = None
                            if " - " in full_text:
                                parts = full_text.split(" - ", 1)
                                title = parts[1].strip() if len(parts) > 1 else None
                            elif ": " in full_text:
                                parts = full_text.split(": ", 1)
                                title = parts[1].strip() if len(parts) > 1 else None
                            elif ":" in full_text:
                                parts = full_text.split(":", 1)
                                title = parts[1].strip() if len(parts) > 1 else None
                            
                            if title and len(title) > 3:
                                # استخراج کلمات کلیدی از تیتر
                                keywords = [
                                    w.strip() 
                                    for w in title.replace("،", " ").replace(".", " ").replace("؛", " ").split() 
                                    if len(w.strip()) > 3
                                ]
                                
                                # اضافه کردن به reverse index
                                for keyword in keywords:
                                    kw_lower = keyword.lower()
                                    reverse_index[kw_lower].append({
                                        "law_name": law_name,
                                        "metadata_type": field_type,
                                        "metadata_field": field_name,
                                        "full_text": full_text,
                                        "title": title,
                                        "collection": collection_name
                                    })
                                
                                # آمار
                                stats[f"{field_name}_titles"] += 1
                
                # ===========================
                # بخش 2: پردازش آراء (UNITY + DADNAMEH)
                # ===========================
                else:
                    # فیلدهای آراء
                    ruling_fields = [
                        ("issuer", "صادرکننده"),
                        ("title", "عنوان رای"),
                        ("category", "حوزه"),
                        ("law_name", "قانون مرتبط"),
                        ("topic", "کلیدواژه")
                    ]
                    
                    for field_name, field_type in ruling_fields:
                        if field_name in payload and payload[field_name]:
                            value = str(payload[field_name]).strip()
                            
                            if not value or len(value) < 3:
                                continue
                            
                            # برای این فیلدها، کل متن را به‌عنوان title استفاده می‌کنیم
                            # (چون معمولاً خودشان یک عبارت کوتاه هستند)
                            title = value
                            
                            # استخراج کلمات کلیدی
                            keywords = [
                                w.strip() 
                                for w in title.replace("،", " ").replace(".", " ").replace("؛", " ").split() 
                                if len(w.strip()) > 3
                            ]
                            
                            # اضافه کردن به reverse index
                            for keyword in keywords:
                                kw_lower = keyword.lower()
                                reverse_index[kw_lower].append({
                                    "law_name": law_name,
                                    "metadata_type": field_type,
                                    "metadata_field": field_name,
                                    "full_text": value,
                                    "title": title,
                                    "collection": collection_name
                                })
                            
                            # آمار
                            if field_name == "issuer":
                                stats["issuer_count"] += 1
                            elif field_name == "title":
                                stats["title_count"] += 1
                            elif field_name == "category":
                                stats["category_count"] += 1
                            elif field_name == "law_name":
                                stats["law_name_count"] += 1
                            elif field_name == "topic":
                                stats["ruling_topic_count"] += 1
            
            batch_count += len(points)
            stats["total_points"] += len(points)
            
            if batch_count % 500 == 0:  # هر 500 سند یک‌بار چاپ کن
                print(f"   پردازش شد: {batch_count} سند...")
            
            if offset is None:
                break
        
        print(f"   ✓ کامل شد: {batch_count} سند از {collection_name}")
    
    print("\n" + "="*80)
    print("✓ استخراج کامل شد:")
    print(f"\n📊 آمار کلی:")
    print(f"  - کل اسناد پردازش شده: {stats['total_points']}")
    print(f"  - کل کلمات کلیدی unique: {len(reverse_index)}")
    
    print(f"\n📚 متادیتای قوانین:")
    print(f"  - تیترهای جلد: {stats['volume_titles']}")
    print(f"  - تیترهای کتاب: {stats['book_titles']}")
    print(f"  - تیترهای باب: {stats['bab_titles']}")
    print(f"  - تیترهای فصل: {stats['chapter_titles']}")
    print(f"  - تیترهای بخش: {stats['section_titles']}")
    print(f"  - تیترهای مبحث: {stats['topic_titles']}")
    print(f"  - تیترهای فقره: {stats['paragraph_titles']}")
    
    print(f"\n⚖️ متادیتای آراء:")
    print(f"  - صادرکنندگان: {stats['issuer_count']}")
    print(f"  - عناوین رای: {stats['title_count']}")
    print(f"  - حوزه‌ها: {stats['category_count']}")
    print(f"  - قوانین مرتبط: {stats['law_name_count']}")
    print(f"  - کلیدواژه‌ها: {stats['ruling_topic_count']}")
    
    print("="*80)
    
    return dict(reverse_index), stats


# اجرای استخراج
print("⏳ شروع استخراج متادیتاها...")
REVERSE_INDEX, STATS = extract_all_metadata_titles()

# ذخیره در فایل
METADATA_INDEX_PATH = r"F:\Thesis\project\2-RAG\vector_store\metadata_reverse_index.json"

with open(METADATA_INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(REVERSE_INDEX, f, ensure_ascii=False, indent=2)

print(f"\n✓ Reverse Index ذخیره شد در: {METADATA_INDEX_PATH}")
print(f"   حجم فایل: {os.path.getsize(METADATA_INDEX_PATH) / 1024:.1f} KB")

# نمایش چند نمونه از هر نوع
print("\n🧪 نمونه‌ای از reverse index:")
print("\n--- نمونه از قوانین ---")
sample_keys = [k for k in list(REVERSE_INDEX.keys())[:20] if any(item['collection'] == COLLECTION_LAWS for item in REVERSE_INDEX[k])][:3]
for key in sample_keys:
    print(f"\nکلیدواژه: '{key}'")
    items = [item for item in REVERSE_INDEX[key] if item['collection'] == COLLECTION_LAWS][:1]
    for item in items:
        print(f"  → {item['metadata_type']}: {item['title'][:50]}...")
        print(f"     قانون: {item['law_name'][:40]}...")

print("\n--- نمونه از آراء ---")
sample_keys = [k for k in list(REVERSE_INDEX.keys())[:50] if any(item['collection'] in [COLLECTION_UNITY, COLLECTION_DADNAMEH] for item in REVERSE_INDEX[k])][:3]
for key in sample_keys:
    print(f"\nکلیدواژه: '{key}'")
    items = [item for item in REVERSE_INDEX[key] if item['collection'] in [COLLECTION_UNITY, COLLECTION_DADNAMEH]][:1]
    for item in items:
        print(f"  → {item['metadata_type']}: {item['title'][:50]}...")
        print(f"     کالکشن: {item['collection']}")


⏳ شروع استخراج متادیتاها...

🔍 در حال پردازش legal_laws...
   پردازش شد: 500 سند...
   پردازش شد: 1000 سند...
   پردازش شد: 1500 سند...
   پردازش شد: 2000 سند...
   پردازش شد: 2500 سند...
   پردازش شد: 3000 سند...
   پردازش شد: 3500 سند...
   پردازش شد: 4000 سند...
   پردازش شد: 4500 سند...
   پردازش شد: 5000 سند...
   پردازش شد: 5500 سند...
   پردازش شد: 6000 سند...
   پردازش شد: 6500 سند...
   پردازش شد: 7000 سند...
   پردازش شد: 7500 سند...
   ✓ کامل شد: 7565 سند از legal_laws

🔍 در حال پردازش legal_votes_unity...
   پردازش شد: 500 سند...
   ✓ کامل شد: 893 سند از legal_votes_unity

🔍 در حال پردازش legal_votes_dadnameh...
   پردازش شد: 500 سند...
   پردازش شد: 1000 سند...
   پردازش شد: 1500 سند...
   پردازش شد: 2000 سند...
   پردازش شد: 2500 سند...
   پردازش شد: 3000 سند...
   پردازش شد: 3500 سند...
   پردازش شد: 4000 سند...
   پردازش شد: 4500 سند...
   پردازش شد: 5000 سند...
   پردازش شد: 5500 سند...
   پردازش شد: 6000 سند...
   پردازش شد: 6500 سند...
   پردازش شد: 7000 سند...
   پر